### Exploratory NB to implement getting ENA data for each study or sample
- I need to do it per sample for sure

In [32]:
from pathlib import Path

# Dataframes and display
import pandas as pd
import numpy as np
from tqdm import tqdm
import requests
import xml.etree.ElementTree as ET
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Warning verbosity
import warnings 
warnings.filterwarnings(action="ignore")

from mgnify_methods.utils.logging import get_logger

logger = get_logger('nb_multiple_studies', level="INFO")
logger.info("Hello from mgnify_methods")
%matplotlib inline 

INFO | nb_multiple_studies | Hello from mgnify_methods


In [2]:
from mgnify_methods.utils.io import (
    load_taxonomy_summary,
    filter_tax_summary,
    assert_taxonomy_integrity,
    filter_number_reads,
)
import mgnify_methods.paper_modules as pm

In [3]:
CONFIG = {
    # === Data Sources ===
    'datasets': {
        'OSD2018': ['MGYS00006608', 'mgnify_data/ERP124424_taxonomy_abundances_SSU_v5.0.tsv'],
        'OSD2019': ['MGYS00006607', 'mgnify_data/ERP124431_taxonomy_abundances_SSU_v5.0.tsv'],
        'Sola': ['MGYS00006680', 'mgnify_data/SRP237882_taxonomy_abundances_SSU_v5.0.tsv'],
        # 'Tara': ['MGYS00000492', 'mgnify_data/ERP003634_taxonomy_abundances_SSU_v5.0.tsv'],
        'Biscay': ['MGYS00006682', 'mgnify_data/SRP334933_taxonomy_abundances_SSU_v5.0.tsv'],
        'Baltic': ['MGYS00006678', 'mgnify_data/ERP140185_taxonomy_abundances_SSU_v5.0.tsv'],
        'BBMO': ['MGYS00006675', 'mgnify_data/ERP122219_taxonomy_abundances_SSU_v5.0.tsv'],
        'Svalbard': ['MGYS00003725', 'mgnify_data/ERP106348_taxonomy_abundances_SSU_v5.0.tsv'],
        'Helgoland': ['MGYS00006686', 'mgnify_data/ERP144826_taxonomy_abundances_SSU_v5.0.tsv'],
        'Fram': ['MGYS00006714', 'mgnify_data/ERP151329_taxonomy_abundances_SSU_v5.0.tsv'],

    },
    # === Output Configuration ===
    'output': {
        'use_timestamp': True,  # Create timestamped output folder
        'cache_dir': 'analysis_cache',  # Cache directory for downloaded data
    },
}

In [4]:
# Fetch analysis metadata
ANALYSIS_CACHE = Path(CONFIG['output']['cache_dir']).resolve()
ds = CONFIG['datasets']

analysis_meta, samples_meta = pm.load_mgnify_meta(ANALYSIS_CACHE, ds)

INFO | mgnify_methods.utils.io | Analysis OSD2018 has 62 samples.
INFO | mgnify_methods.utils.io | Analysis OSD2019 has 48 samples.
INFO | mgnify_methods.utils.io | Analysis Sola has 283 samples.
INFO | mgnify_methods.utils.io | Analysis Biscay has 52 samples.
INFO | mgnify_methods.utils.io | Analysis Baltic has 665 samples.
INFO | mgnify_methods.utils.io | Analysis BBMO has 249 samples.
INFO | mgnify_methods.utils.io | Analysis Svalbard has 159 samples.
INFO | mgnify_methods.utils.io | Analysis Helgoland has 357 samples.
INFO | mgnify_methods.utils.io | Analysis Fram has 205 samples.
INFO | mgnify_methods.utils.io | Analysis OSD2019 has 48 samples.
INFO | mgnify_methods.utils.io | Analysis Sola has 283 samples.
INFO | mgnify_methods.utils.io | Analysis Biscay has 52 samples.
INFO | mgnify_methods.utils.io | Analysis Baltic has 665 samples.
INFO | mgnify_methods.utils.io | Analysis BBMO has 249 samples.
INFO | mgnify_methods.utils.io | Analysis Svalbard has 159 samples.
INFO | mgnify_m

In [23]:
def parse_sample_xml(xml_bytes):
    root = ET.fromstring(xml_bytes)

    rows = []

    for sample in root.findall(".//SAMPLE"):
        row = {}

        # --- basic metadata ---
        row["accession"] = sample.attrib.get("accession")
        row["alias"] = sample.attrib.get("alias")
        row["center_name"] = sample.attrib.get("center_name")

        row["title"] = sample.findtext("TITLE")
        row["description"] = sample.findtext("DESCRIPTION")

        # --- identifiers ---
        row["primary_id"] = sample.findtext(".//PRIMARY_ID")
        row["secondary_id"] = sample.findtext(".//SECONDARY_ID")

        # --- taxonomy ---
        row["taxon_id"] = sample.findtext(".//TAXON_ID")
        row["scientific_name"] = sample.findtext(".//SCIENTIFIC_NAME")

        # --- attributes (key-value flattening) ---
        for attr in sample.findall(".//SAMPLE_ATTRIBUTE"):
            tag = attr.findtext("TAG")
            value = attr.findtext("VALUE")
            units = attr.findtext("UNITS")

            if tag:
                col = tag.strip()

                # optionally append units
                if units:
                    row[col] = value
                    row[f"{col}_units"] = units
                else:
                    row[col] = value

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
def retrieve_ena_metadata(samples_meta):
    dfs = []
    for biosample_accession in samples_meta['biosample'].unique()[:5]:
        time.sleep(0.5)  # Be polite to the ENA API
        url = f"https://www.ebi.ac.uk/ena/browser/api/xml/{biosample_accession}"
        response = requests.get(url)
    
        if response.status_code == 200:
            logger.info(f"Successfully retrieved metadata for {biosample_accession}")
            dfs.append(parse_sample_xml(response.content))
        else:
            logger.error(f"Failed to retrieve metadata for {biosample_accession} (Status code: {response.status_code})")
    return pd.concat(dfs, ignore_index=True)

def smart_numeric_cast(df, threshold=1.0):
    for col in df.columns:
        converted = pd.to_numeric(df[col], errors="coerce")
        success_rate = converted.notna().mean()

        if success_rate >= threshold:
            df[col] = converted

    return df

INFO | nb_multiple_studies | Successfully retrieved metadata for SAMEA7392422
INFO | nb_multiple_studies | Successfully retrieved metadata for SAMEA7392426
INFO | nb_multiple_studies | Successfully retrieved metadata for SAMEA7392429
INFO | nb_multiple_studies | Successfully retrieved metadata for SAMEA7392430
INFO | nb_multiple_studies | Successfully retrieved metadata for SAMEA7392434
INFO | nb_multiple_studies | Successfully retrieved metadata for SAMEA7392435
INFO | nb_multiple_studies | Successfully retrieved metadata for SAMEA7392440
INFO | nb_multiple_studies | Successfully retrieved metadata for SAMEA7392447
INFO | nb_multiple_studies | Successfully retrieved metadata for SAMEA7392452
INFO | nb_multiple_studies | Successfully retrieved metadata for SAMEA7392462


KeyboardInterrupt: 

In [ ]:
df = retrieve_ena_metadata(samples_meta)
df = smart_numeric_cast(df)
df.head()

,accession,alias,center_name,title,description,primary_id,secondary_id,taxon_id,scientific_name,Sampling Site,Temperature,Temperature_units,environment (material),sample_description,event_method,water environmental package,Depth,Depth_units,sample_treatment_storage,ENA-FIRST-PUBLIC,sample_treatment_chemicals,ENA-LAST-UPDATE,sample_size-fraction_upper-threshold,environment (biome),environment (feature),event_device,sample_content,sample_filtration time (min),ENA-CHECKLIST,submitted to insdc,organism,environmental package,Latitude Start,Latitude Start_units,Sampling Platform,Longitude Start,Longitude Start_units,event_comment,Salinity,Salinity_units,Marine Region,project name,sample_size-fraction_lower-threshold,sequencing method,Protocol Label,Event Date/Time,sample_container,sample_quantity (l),Sampling Campaign,sampling_objective
0,SAMEA7392422,32513:f016dd88-3b2f-48a1-b740-b5a51edf1232,Ocean Sampling Day Consortium,OSD232_2018-06_1_NPL022,surface water sample from the Tyrrhenian Sea,SAMEA7392422,ERS5150811,408172,marine metagenome,"OSD232, Portici Italy",25.00,ºC,sea water [ENVO:00002149],surface water sample from the Tyrrhenian Sea,sampling water from a boat,water,0,m,-80,2021-03-04,None,2021-10-01,not provided,marine biome [ENVO:00000447],microbial community [PCO:1000004],Boat,seawater,110,ERC000027,true,marine metagenome,water,40.80,DD,not provided,14.34,DD,water mass very clear. Low content from NW,36.80,psu,https://www.marineregions.org/gazetteer.php?p=details&id=25611,Ocean Sampling Day,0.22,Illumina MiSeq,NPL022,2018-06-21T10:40:00Z/2018-06-21T17:30:00Z,Tank,4.00,OSD-Jun-2018,OSD-232-PROK
1,SAMEA7392426,32509:f016dd88-3b2f-48a1-b740-b5a51edf1232,Ocean Sampling Day Consortium,OSD227_2018-06_1_NPL022,surface water sample from the English Channel,SAMEA7392426,ERS5150815,408172,marine metagenome,"OSD227, GO SBR Crenula France",14.74,ºC,sea water [ENVO:00002149],surface water sample from the English Channel,not provided,water,1,m,-80,2021-03-04,None,2021-10-01,No prefiltration,estuarine biome [ENVO:01000020],microbial community [PCO:1000004],Hand pump,particulate matter,30,ERC000027,true,marine metagenome,water,48.67,DD,RV Neomysis,-3.89,DD,NE wind 25-30km/h sunny all filters are coloured light brown time of filtration (UTC) 13:10,34.70,psu,https://www.marineregions.org/gazetteer.php?p=details&id=2389,Ocean Sampling Day,0.22,Illumina MiSeq,NPL022,2018-06-21T11:40:00Z/2018-06-21T11:46:00Z,Sterivex,2.76,OSD-Jun-2018,not provided
